In [5]:
#!/usr/bin/env python3
"""
HK Address Parser - Final App with Integrated Evaluation
Evaluates address splitting logic, calculates split-aware confidence, 
and measures Accuracy relative to Confidence Thresholds.
Includes interchangeable logic for Estate Name and Building Name.
"""

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./xlm_roberta_large_crfV2"
LOG_FILE = "address_split_results_xlm_largeV2.log"
TEST_FILE = "data/test.jsonl"
MAX_LEN = 128
BATCH_SIZE = 32

THRESHOLDS = [0.0, 0.2, 0.3, 0.4, 0.50, 0.60, 0.70, 0.80]
ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ==========================================
# UTILITY FUNCTIONS FOR EVALUATION
# ==========================================
def flatten_json(output_dict):
    """Flattens the nested ground truth JSON from test.jsonl."""
    flat = {}
    if "line1" in output_dict and isinstance(output_dict["line1"], dict):
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def normalize_for_eval(text):
    """Lowercases and removes ALL spaces/punctuation for strict string comparison."""
    if not text:
        return ""
    # Remove spaces and common punctuation for evaluation purposes
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, :] = 0.0
        self.crf.transitions.data[:, o_id] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]
            self.crf.start_transitions.data[tid] = 0.0
            self.crf.end_transitions.data[tid] = 0.0

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction="mean")
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS
# ==========================================
class HKAddressParser:
    def __init__(self, model_path, max_len=128):
        self.model_path = model_path
        self.max_len = max_len
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        self.config = AutoConfig.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        label_list = [self.config.id2label[k] if isinstance(k, int) else self.config.id2label[str(k)] 
                      for k in sorted([int(k) for k in self.config.id2label.keys()])]

        self.model = BertCRFForTokenClassification(self.config, model_path, label_list)

        weights_path = os.path.join(model_path, "pytorch_model.bin")
        if os.path.exists(weights_path):
            self._load_weights_with_progress(weights_path)
        else:
            st_path = os.path.join(model_path, "model.safetensors")
            if os.path.exists(st_path):
                self._load_weights_with_progress(st_path, safetensors=True)

        self.model.to(self.device)
        self.model.eval()

    def _load_weights_with_progress(self, weights_path, safetensors=False):
        print(f"📦 Loading weights from {weights_path} ...")
        if safetensors:
            from safetensors.torch import load_file
            state_dict = load_file(weights_path, device="cpu")
        else:
            state_dict = torch.load(weights_path, map_location="cpu")

        own = self.model.state_dict()
        for name in list(own.keys()):
            if name in state_dict and own[name].shape == state_dict[name].shape:
                own[name].copy_(state_dict[name])

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available(): return torch.device("cpu")
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id, max_free_mb = 0, 0
            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id, free_memory, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
                if gpu_util < 30 and free_memory > max_free_mb:
                    max_free_mb, best_id = free_memory, gpu_id
            return torch.device(f"cuda:{best_id}")
        except Exception: return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)

        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output, conf_output = {}, {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        """
        Splits address and calculates confidence strictly based on labels 
        that influenced the split decision. Includes logic to correctly bind 
        Building Numbers with Village and Street names, and separates Estate/Phase 
        from Building/Block when both hierarchical levels exist.
        """
        line1_keys = set()
        logic_keys_used = set()
        
        # 1. Micro elements
        if "flat" in extracted_labels: 
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels: 
            line1_keys.add("floor")
            logic_keys_used.add("floor")

        # 2. Structural elements (Hierarchical separation logic)
        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels
        
        # RULE 1: If Building OR Block exists, they take priority for Line 1 (micro).
        # Estate and Phase are omitted from line1_keys so they drop to Line 2 (macro).
        if has_bldg or has_block:
            if has_bldg:
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
            if has_block:
                line1_keys.add("block")
                logic_keys_used.add("block")
                
            # Track Estate/Phase for confidence calculation, but leave them OUT of line1_keys
            # if has_est:
            #     logic_keys_used.add("estate_name")
            # if has_phase:
            #     logic_keys_used.add("phase")
                
        # RULE 2: If NO Building or Block, but Estate exists, Estate acts as the building (Line 1).
        elif has_est:
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase: 
                line1_keys.add("phase")
                logic_keys_used.add("phase")
                
        # RULE 3: Fallbacks for Village / Street / Building Number
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")

        # 3. Assign tokens
        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        token_groups = []
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys: token_groups.append("micro")
            elif mapped_tag != "o" and mapped_tag != "O": token_groups.append("macro")
            else: token_groups.append("O") 
                
        # Resolve punctuation
        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro" 
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)
                
        # Group adjacent tokens into segments by their tag
        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None
        
        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]
            
            if group == "micro":
                if tag != 'o' and tag != 'O':
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg: curr_micro_seg.append(word)
                    else: curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag != 'o' and tag != 'O':
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg: curr_macro_seg.append(word)
                    else: curr_macro_seg, curr_macro_tag = [word], "o"
                        
        if curr_micro_seg: micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg: macro_segments.append((curr_macro_tag, curr_macro_seg))
            
        def reorder_segments(segments, is_chinese):
            """Moves the building_number chunk to sit directly next to the primary grouping element"""
            bldg_no_idx = next((i for i, s in enumerate(segments) if s[0] == "building_number"), -1)
            if bldg_no_idx == -1: return segments
            
            bldg_no_seg = segments.pop(bldg_no_idx)
            target_tags = ["village_name", "street_name", "estate_name", "building_name"]
            
            if is_chinese:
                # In Chinese, Building Number goes immediately AFTER Village/Street
                target_idx = -1
                for i, s in enumerate(segments):
                    if s[0] in target_tags: target_idx = i
                if target_idx != -1: segments.insert(target_idx + 1, bldg_no_seg)
                else: segments.insert(0, bldg_no_seg)
            else:
                # In English, Building Number goes immediately BEFORE Village/Street
                target_idx = -1
                for i, s in enumerate(segments):
                    if s[0] in target_tags:
                        target_idx = i
                        break
                if target_idx != -1: segments.insert(target_idx, bldg_no_seg)
                else: segments.append(bldg_no_seg)
            return segments

        # Apply reordering for both output lines to standardize formatting
        micro_segments = reorder_segments(micro_segments, is_chinese)
        macro_segments = reorder_segments(macro_segments, is_chinese)

        micro_string = "".join("".join(words) for tag, words in micro_segments)
        macro_string = "".join("".join(words) for tag, words in macro_segments)
        
        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()

        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)
        
        # Calculate Logic Confidence (Now strictly uses verified keys)
        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)
            
        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ["district", "region", "sub_district"]: split_conf *= c

        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string

        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, full_addresses, batch_size=32):
        all_results = []
        for i in tqdm(range(0, len(full_addresses), batch_size), desc="Processing"):
            batch = full_addresses[i : i + batch_size]
            
            encoded = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=self.max_len, return_offsets_mapping=True, return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)
            batch_offsets = encoded["offset_mapping"].tolist()

            with torch.no_grad():
                prediction_ids_batch, emissions_batch = self.model(input_ids=input_ids, attention_mask=attention_mask)

            for idx, address_str in enumerate(batch):
                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                offsets = batch_offsets[idx]
                prediction_ids = prediction_ids_batch[idx]
                token_probs = torch.softmax(emissions_batch[idx], dim=-1)

                for j, tag_id in enumerate(prediction_ids):
                    start, end = offsets[j]
                    if start == end: continue

                    tag = self.config.id2label[tag_id] if isinstance(tag_id, int) else self.config.id2label[str(tag_id)]
                    conf = token_probs[j, int(tag_id)].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)

                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"

                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)
                
                # Split and get logic-aware confidence
                line1, line2, split_conf, used_keys = self._split_address(extracted_mapped, address_str, parsed_entities, conf_mapped)

                all_results.append((address_str, extracted_mapped, conf_mapped, line1, line2, split_conf, used_keys))

        return all_results


# ==========================================
# MAIN EXECUTION & EVALUATION LOOP
# ==========================================
def main():
    if not os.path.exists(MODEL_DIR) or not os.path.exists(TEST_FILE):
        print("❌ Error: Model directory or Test file not found.")
        return

    parser = HKAddressParser(model_path=MODEL_DIR, max_len=MAX_LEN)

    print(f"🚀 Loading dataset from {TEST_FILE}...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        test_data = [json.loads(line) for line in file if line.strip() and not line.startswith("#")]
        
    inputs = [item["input"].strip() for item in test_data]
    
    start_time = time.perf_counter()
    results = parser.parse_batch(inputs, batch_size=BATCH_SIZE)
    total_time = time.perf_counter() - start_time

    # Initialize New Splitting Metrics Structure
    stats = {
        t: {
            "total_in_bin": 0,
            "logic_correct": 0,
            "line1_correct": 0,
            "line2_correct": 0,
            "full_correct": 0
        } for t in THRESHOLDS
    }

    excluded_count = 0

    print(f"✍️ Evaluating and writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (address, pred_tags, pred_confs, line1, line2, split_conf, used_keys) in enumerate(results):
            ground_truth_flat = flatten_json(test_data[idx].get("output", {}))
            
            # --- CHECK FOR CORRUPTED GROUND TRUTH ---
            # Exclude if ground truth has labels not found inside the input text
            norm_address = normalize_for_eval(address)
            is_corrupted = False
            for _, gt_val in ground_truth_flat.items():
                norm_gt_val = normalize_for_eval(gt_val)
                if norm_gt_val and norm_gt_val not in norm_address:
                    is_corrupted = True
                    break
            
            if is_corrupted:
                excluded_count += 1
                log.write(f"--- Result {idx + 1} [EXCLUDED: CORRUPTED DATA] ---\n")
                log.write(f"Original Input  : {address}\n")
                log.write("Reason          : Ground truth contains values not present in the input text.\n")
                log.write("-" * 50 + "\n")
                continue

            # --- EVALUATE CORRECTNESS OF THE PREDICTION ---
            pred_ff = normalize_for_eval(pred_tags.get("floor", "") + pred_tags.get("flat", ""))
            gt_ff = normalize_for_eval(ground_truth_flat.get("floor", "") + ground_truth_flat.get("flat", ""))
            
            p_bldg = normalize_for_eval(pred_tags.get("building_name", ""))
            p_est  = normalize_for_eval(pred_tags.get("estate_name", ""))
            g_bldg = normalize_for_eval(ground_truth_flat.get("building_name", ""))
            g_est  = normalize_for_eval(ground_truth_flat.get("estate_name", ""))
            
            field_correct = {}
            for field in ALL_FIELDS:
                p_norm = normalize_for_eval(pred_tags.get(field, ""))
                g_norm = normalize_for_eval(ground_truth_flat.get(field, ""))
                
                if p_norm == g_norm:
                    field_correct[field] = True
                elif field in ['floor', 'flat'] and pred_ff == gt_ff and pred_ff != "":
                    field_correct[field] = True
                elif field in ['building_name', 'estate_name'] and (p_bldg == g_est and p_est == g_bldg):
                    # Evaluates to True if Building Name and Estate Name were perfectly swapped
                    field_correct[field] = True
                else:
                    field_correct[field] = False
                    
            # 1. Split Logic Determination Correctness
            is_logic_correct = all(field_correct.get(k, False) for k in used_keys) if used_keys else True
            
            # 2. Line 1 Exact Match (Micro fields)
            line1_fields = ["flat", "floor", "block", "phase", "building_name", "estate_name"]
            is_l1_correct = all(field_correct[f] for f in line1_fields)
            
            # 3. Line 2 Exact Match (Macro fields)
            line2_fields = ["village_name", "building_number", "street_name", "sub_district", "district", "region"]
            is_l2_correct = all(field_correct[f] for f in line2_fields)
            
            # 4. Full Address Exact Match
            is_full_correct = is_l1_correct and is_l2_correct

            # --- BIN THE ACCURACY BY CONFIDENCE ---
            for t in THRESHOLDS:
                if split_conf >= t:
                    stats[t]["total_in_bin"] += 1
                    if is_logic_correct: stats[t]["logic_correct"] += 1
                    if is_l1_correct: stats[t]["line1_correct"] += 1
                    if is_l2_correct: stats[t]["line2_correct"] += 1
                    if is_full_correct: stats[t]["full_correct"] += 1

            # --- WRITE FULL LOGS ---
            log.write(f"--- Result {idx + 1} ---\n")
            log.write(f"Original Input  : {address}\n")
            log.write(f"Logic Keys Used : {used_keys}\n")
            log.write(f"Split Conf      : {split_conf:.6f} (Product of keys used in split)\n")
            log.write(f"Output Line 1   : {line1}\n")
            log.write(f"Output Line 2   : {line2}\n")
            
            if is_full_correct: log.write("✅ EXACT MATCH\n")
            else: log.write("❌ MISMATCH FOUND\n")
            
            for field in ALL_FIELDS:
                pred_val = pred_tags.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                if pred_val or gt_val:
                    status = "✅" if field_correct[field] else "❌"
                    conf_str = f"  conf={pred_confs.get(field, 0.0):.4f}"
                    log.write(f" {status} {field.upper()}:{conf_str}\n")
                    log.write(f"   PRED: {pred_val if pred_val else '[None]'}\n")
                    log.write(f"   TRUE: {gt_val if gt_val else '[None]'}\n")
                    
            log.write("-" * 50 + "\n")

        # --- WRITE SUMMARY TABLE AT THE BOTTOM OF THE LOG ---
        total_samples = len(inputs)
        valid_samples = total_samples - excluded_count
        
        def build_table_str():
            out = "\n" + "=" * 65 + "\n"
            out += "📊 ADDRESS SPLITTING METRICS (CALIBRATED ACCURACY)\n"
            out += "=" * 65 + "\n"
            out += f"Total Addresses Provided: {total_samples}\n"
            out += f"Excluded (Corrupted)    : {excluded_count}\n"
            out += f"Total Valid Evaluated   : {valid_samples}\n"
            out += f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n\n"

            for t in THRESHOLDS:
                title = "ALL PREDICTIONS (Threshold 0%)" if t == 0.0 else f"PREDICTIONS WITH ≥ {int(t*100)}% CONFIDENCE"
                out += "-" * 65 + "\n"
                out += f"🚀 {title}\n"
                out += "-" * 65 + "\n"
                
                total_in_bin = stats[t]["total_in_bin"]
                if total_in_bin > 0:
                    out += f"{'METRIC':<35} | {'ACCURACY (Correct / Total in Bin)'}\n"
                    out += "-" * 65 + "\n"
                    
                    l_acc = (stats[t]["logic_correct"] / total_in_bin) * 100
                    l1_acc = (stats[t]["line1_correct"] / total_in_bin) * 100
                    l2_acc = (stats[t]["line2_correct"] / total_in_bin) * 100
                    full_acc = (stats[t]["full_correct"] / total_in_bin) * 100
                    
                    out += f"{'Split Logic Determination Correct':<35} | {l_acc:>6.2f}%  ({stats[t]['logic_correct']}/{total_in_bin})\n"
                    out += f"{'Line 1 (Micro) Components Correct':<35} | {l1_acc:>6.2f}%  ({stats[t]['line1_correct']}/{total_in_bin})\n"
                    out += f"{'Line 2 (Macro) Components Correct':<35} | {l2_acc:>6.2f}%  ({stats[t]['line2_correct']}/{total_in_bin})\n"
                    out += f"{'Full Address Perfect Match':<35} | {full_acc:>6.2f}%  ({stats[t]['full_correct']}/{total_in_bin})\n\n"
                else:
                    out += f"No samples met the >= {int(t*100)}% confidence threshold.\n\n"
            return out

        table_output = build_table_str()
        print(table_output)
        log.write(table_output)

if __name__ == "__main__":
    main()

DEBUG: Using Device -> cuda:2


Loading weights: 0it [00:00, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: ./xlm_roberta_large_crfV2
Key                                                           | Status     | 
--------------------------------------------------------------+------------+-
bert.encoder.layer.{0...23}.output.LayerNorm.weight           | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.LayerNorm.bias   | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.key.bias           | UNEXPECTED | 
bert.encoder.layer.{0...23}.intermediate.dense.weight         | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.LayerNorm.weight | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.query.bias         | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.dense.weight     | UNEXPECTED | 
bert.encoder.layer.{0...23}.output.LayerNorm.bias             | UNEXPECTED | 
bert.encoder.layer.{0...23}.output.dense.bias                 | UNEXPECTED | 
crf.end_transitions                                           | UNE

📦 Loading weights from ./xlm_roberta_large_crfV2/pytorch_model.bin ...
🚀 Loading dataset from data/test.jsonl...


Processing:   0%|          | 0/647 [00:00<?, ?it/s]

✍️ Evaluating and writing results to address_split_results_xlm_largeV2.log...

📊 ADDRESS SPLITTING METRICS (CALIBRATED ACCURACY)
Total Addresses Provided: 20696
Excluded (Corrupted)    : 394
Total Valid Evaluated   : 20302
⏱️ Total Inference runtime: 134.2759 seconds

-----------------------------------------------------------------
🚀 ALL PREDICTIONS (Threshold 0%)
-----------------------------------------------------------------
METRIC                              | ACCURACY (Correct / Total in Bin)
-----------------------------------------------------------------
Split Logic Determination Correct   |  97.47%  (19788/20302)
Line 1 (Micro) Components Correct   |  97.33%  (19760/20302)
Line 2 (Macro) Components Correct   |  97.10%  (19713/20302)
Full Address Perfect Match          |  94.85%  (19257/20302)

-----------------------------------------------------------------
🚀 PREDICTIONS WITH ≥ 20% CONFIDENCE
-----------------------------------------------------------------
METRIC         

In [1]:
#!/usr/bin/env python3
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CUSTOM ARCHITECTURE (Updated for XLM-R Large + Constraints)
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        num_tags = len(label_list)
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, :] = 0.0
        self.crf.transitions.data[:, o_id] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]
            self.crf.start_transitions.data[tid] = 0.0
            self.crf.end_transitions.data[tid] = 0.0

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction="mean")
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS
# ==========================================
class HKAddressParser:
    def __init__(self, model_path, max_len=128):
        self.model_path = model_path
        self.max_len = max_len
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        self.config = AutoConfig.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

        # Dynamically build label list for constraints
        label_list = [self.config.id2label[k] if isinstance(k, int) else self.config.id2label[str(k)] 
                      for k in sorted([int(k) for k in self.config.id2label.keys()])]

        self.model = BertCRFForTokenClassification(self.config, model_path, label_list)

        weights_path = os.path.join(model_path, "pytorch_model.bin")
        if os.path.exists(weights_path):
            self._load_weights_with_progress(weights_path)
        else:
            st_path = os.path.join(model_path, "model.safetensors")
            if os.path.exists(st_path):
                self._load_weights_with_progress(st_path, safetensors=True)
            else:
                print(f"⚠️ Warning: Weights not found at {weights_path}")

        self.model.to(self.device)
        self.model.eval()

    def _load_weights_with_progress(self, weights_path, safetensors=False):
        print(f"📦 Loading weights from {weights_path} ...")
        if safetensors:
            from safetensors.torch import load_file
            state_dict = load_file(weights_path, device="cpu")
        else:
            state_dict = torch.load(weights_path, map_location="cpu")

        model_keys = set(self.model.state_dict().keys())
        loaded = 0
        missing_in_ckpt = []
        with tqdm(total=len(model_keys), desc="Loading weights", unit="tensor") as pbar:
            own = self.model.state_dict()
            for name in list(own.keys()):
                if name in state_dict and own[name].shape == state_dict[name].shape:
                    own[name].copy_(state_dict[name])
                    loaded += 1
                else:
                    missing_in_ckpt.append(name)
                pbar.update(1)
                pbar.set_postfix(loaded=loaded)

        unexpected = [k for k in state_dict.keys() if k not in model_keys]
        print(
            f"✅ Loaded {loaded}/{len(model_keys)} tensors | "
            f"missing={len(missing_in_ckpt)} unexpected={len(unexpected)}"
        )

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available():
            return torch.device("cpu")
        try:
            print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
                 "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id = -1
            max_free_mb = 0
            fallback_id = 0
            fallback_max_mb = 0

            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id = int(parts[0])
                free_memory = int(parts[1])
                gpu_util = int(parts[2])

                if free_memory > fallback_max_mb:
                    fallback_max_mb = free_memory
                    fallback_id = gpu_id

                if gpu_util < 30:
                    if free_memory > max_free_mb:
                        max_free_mb = free_memory
                        best_id = gpu_id

            if best_id != -1:
                return torch.device(f"cuda:{best_id}")
            else:
                return torch.device(f"cuda:{fallback_id}")
        except Exception as e:
            return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)
        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output = {}
        conf_output = {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        """
        Splits address and calculates confidence strictly based on labels 
        that influenced the split decision. Includes logic to correctly bind 
        Building Numbers with Village and Street names, and separates Estate/Phase 
        from Building/Block when both hierarchical levels exist.
        """
        line1_keys = set()
        logic_keys_used = set()
        
        # 1. Micro elements
        if "flat" in extracted_labels: 
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels: 
            line1_keys.add("floor")
            logic_keys_used.add("floor")

        # 2. Structural elements (Hierarchical separation logic)
        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels
        
        # RULE 1: If Building OR Block exists, they take priority for Line 1 (micro).
        # Estate and Phase are omitted from line1_keys so they drop to Line 2 (macro).
        if has_bldg or has_block:
            if has_bldg:
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
            if has_block:
                line1_keys.add("block")
                logic_keys_used.add("block")
                
            # Track Estate/Phase for confidence calculation, but leave them OUT of line1_keys
            # if has_est:
            #     logic_keys_used.add("estate_name")
            # if has_phase:
            #     logic_keys_used.add("phase")
                
        # RULE 2: If NO Building or Block, but Estate exists, Estate acts as the building (Line 1).
        elif has_est:
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase: 
                line1_keys.add("phase")
                logic_keys_used.add("phase")
                
        # RULE 3: Fallbacks for Village / Street / Building Number
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no:
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")

        # 3. Assign tokens
        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        token_groups = []
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys: token_groups.append("micro")
            elif mapped_tag != "o" and mapped_tag != "O": token_groups.append("macro")
            else: token_groups.append("O") 
                
        # Resolve punctuation
        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro" 
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)
                
        # Group adjacent tokens into segments by their tag
        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None
        
        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]
            
            if group == "micro":
                if tag != 'o' and tag != 'O':
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg: curr_micro_seg.append(word)
                    else: curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag != 'o' and tag != 'O':
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg: curr_macro_seg.append(word)
                    else: curr_macro_seg, curr_macro_tag = [word], "o"
                        
        if curr_micro_seg: micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg: macro_segments.append((curr_macro_tag, curr_macro_seg))
            
        def reorder_segments(segments, is_chinese):
            """Moves the building_number chunk to sit directly next to the primary grouping element"""
            bldg_no_idx = next((i for i, s in enumerate(segments) if s[0] == "building_number"), -1)
            if bldg_no_idx == -1: return segments
            
            bldg_no_seg = segments.pop(bldg_no_idx)
            target_tags = ["village_name", "street_name", "estate_name", "building_name"]
            
            if is_chinese:
                # In Chinese, Building Number goes immediately AFTER Village/Street
                target_idx = -1
                for i, s in enumerate(segments):
                    if s[0] in target_tags: target_idx = i
                if target_idx != -1: segments.insert(target_idx + 1, bldg_no_seg)
                else: segments.insert(0, bldg_no_seg)
            else:
                # In English, Building Number goes immediately BEFORE Village/Street
                target_idx = -1
                for i, s in enumerate(segments):
                    if s[0] in target_tags:
                        target_idx = i
                        break
                if target_idx != -1: segments.insert(target_idx, bldg_no_seg)
                else: segments.append(bldg_no_seg)
            return segments

        # Apply reordering for both output lines to standardize formatting
        micro_segments = reorder_segments(micro_segments, is_chinese)
        macro_segments = reorder_segments(macro_segments, is_chinese)

        micro_string = "".join("".join(words) for tag, words in micro_segments)
        macro_string = "".join("".join(words) for tag, words in macro_segments)
        
        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()

        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)
        
        # Calculate Logic Confidence (Now strictly uses verified keys)
        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)
            
        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ["district", "region", "sub_district"]: split_conf *= c

        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string

        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, address_pairs, batch_size=32):
        all_results = []
        for i in range(0, len(address_pairs), batch_size):
            batch = address_pairs[i : i + batch_size]
            full_addresses = []
            for p1, p2 in batch:
                parts = [p.strip() for p in (p1, p2) if p.strip()]
                full_addresses.append(" ".join(parts) if parts else "")

            encoded = self.tokenizer(
                full_addresses,
                padding=True,
                truncation=True,
                max_length=self.max_len,
                return_offsets_mapping=True,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)
            batch_offsets = encoded["offset_mapping"].tolist()

            try:
                with torch.no_grad():
                    prediction_ids_batch, emissions_batch = self.model(
                        input_ids=input_ids, attention_mask=attention_mask
                    )
            except Exception as e:
                # Appends 8 fields as expected by the new unpack format
                all_results.extend([("", "", "", {}, {}, 0.0, [], f"ERROR: {str(e)}")] * len(batch))
                continue

            for idx, address_str in enumerate(full_addresses):
                if not address_str:
                    all_results.append(("", "", "", {}, {}, 0.0, [], "EMPTY_INPUT"))
                    continue

                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                offsets = batch_offsets[idx]
                prediction_ids = prediction_ids_batch[idx]
                token_probs = torch.softmax(emissions_batch[idx], dim=-1)

                for j, tag_id in enumerate(prediction_ids):
                    start, end = offsets[j]
                    if start == end: continue

                    if isinstance(tag_id, int):
                        tag = self.config.id2label[tag_id]
                        conf = token_probs[j, tag_id].item()
                    else:
                        tag = self.config.id2label[str(tag_id)]
                        conf = token_probs[j, int(tag_id)].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)

                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "")
                    if tag == "O": entity_group = "O"

                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)
                
                # Split and get logic-aware confidence
                line1, line2, split_conf, used_keys = self._split_address(extracted_mapped, address_str, parsed_entities, conf_mapped)

                all_results.append((
                    address_str, line1, line2,
                    extracted_mapped, conf_mapped, split_conf, used_keys, "SUCCESS"
                ))

        return all_results

    def parse(self, address_part1, address_part2=""):
        result_list = self.parse_batch([(address_part1, address_part2)], batch_size=1)
        return result_list[0]

# ==========================================
# TESTING SCRIPT
# ==========================================
if __name__ == "__main__":
    MODEL_DIR = "./xlm_roberta_large_checkpointV2"

    print(f"Loading model from {MODEL_DIR}...")
    try:
        parser = HKAddressParser(model_path=MODEL_DIR)

        test_cases = [
            ("1M 16 lung sum avenue sheung shui north N.T", ""),
            ("27LD, Block 5, Hemera, Lohas Park, Lohas Road 1, TKO, HK", ""),
            ("21A / 5th Fl., Metroplaza Tower A, Nº 223 Hing Fong Road, Kwai Fong, N.T.", ""),
            ("21A / 5th Fl., Metroplaza Tower 1, 223 Nº Hing Fong Road, Kwai Fong, N.T.", ""),
            ("21A / 5th Fl., Metroplaza Tower A, NW 223 Hing Fong Road, Kwai Fong, N.T.", ""),
            ("21A / 5th Fl., Metroplaza Tower 1, 223 NW Hing Fong Road, Kwai Fong, N.T.", ""),
            ("長洲東灣東堤小築彌敦道一百二十三12座H室5樓", ""),
            ("深水埗白田街123至125白田邨1号楼, ４０４室Part 1", ""),
            ("深水埗中正路二段123巷45弄67號白田邨1號樓廿樓, ４０４室Part 1", ""),
            ("深水埗南京路123号之2甲白田邨第1幢第5樓", ""),
            ("深水埗白田街白田邨A幢１st flor, ４０４室Part 1", ""),
            ("馬鞍山西沙路港鐵站迎海1座15樓A栋, ４０４室Part 1", ""),
            ("深水埗白田街白田邨壹號樓壹樓, ４０４室Part 1", ""),
            ("深水埗白田街白田邨一號樓一樓, ４０４室Part 1", ""),
            ("深水埗白田街白田邨第1幢第5樓", ""),
            ("香港仔中心香港仔大道港興閣", "第88座第五樓"),
            ("香港 西灣河 東苑 Block Number 8 Floor Two A", ""),
            ("香港 西灣河 東苑 Storey 4 B", ""),
            ("room8100, fu keng house, tai wo hau estate Second Flr", ""),
            ("大埔中心 座数: 1 二字樓 b", ""),
            ("大澳丈量約份第310約First Floor b", ""),
            ("Flat A, UG, Happy Mansion, 28 Lockhart Road, Wan Chai, Hong Kong", ""),
            ("Unit 7B, M/F, Harbour View Tower, 88 Gloucester Road, Causeway Bay", ""),
            ("Room 1503, LG1, Sunshine Court, 156 Nathan Road, Tsim Sha Tsui, Kowloon", ""),
            ("9C, 5-F, The Grandiose, 9 Tong Chun Street, Tseung Kwan O, New Territories", ""),
            ("Flat 3, Fifth Floor,Block Number 8, Parkview Mansion, 88 Tai Hang Road, Tai Hang, Hong Kong Island", ""),
            ("21A / 5th Fl., Metroplaza T-A, 223 Hing Fong Road, Kwai Fong, N.T.", ""),
            ("1602, 5th Flr, City One Plaza, 1 Ngan Shing Street, Sha Tin, New Territories", ""),
            ("Rm 5A, Fl. 5, Bk A, Lucky Building, 45 Queen’s Road Central, Central, HK", ""),
        ]

        print("\nRunning BATCH Test:\n" + "=" * 70)
        batch_results = parser.parse_batch(test_cases, batch_size=2)

        for i, (a1, a2) in enumerate(test_cases):
            address_str, l1, l2, tags, confs, overall, used_keys, status = batch_results[i]

            print(f"Original Input : '{address_str}'")
            print(f"Line 1         : {l1}")
            print(f"Line 2         : {l2}")
            print(f"Extracted Tags : {tags}")
            print(f"Confidences    : { {k: round(v, 4) for k, v in confs.items()} }")
            print(f"Logic Keys Used: {used_keys}")
            print(f"Overall Split C: {overall:.6f}   (split logic-aware product)")
            print(f"Status         : {status}")
            print("-" * 70)

    except FileNotFoundError as e:
        print(f"❌ Initialization Failed: {e}")
        print("Please ensure your model directory exists before running the test cases.")

Loading model from ./xlm_roberta_large_checkpointV2...

🔍 Scanning available GPUs safely via nvidia-smi...
DEBUG: Using Device -> cuda:1


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.1+cu118).


Loading weights: 0it [00:00, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: ./xlm_roberta_large_checkpointV2
Key                                                           | Status     | 
--------------------------------------------------------------+------------+-
bert.encoder.layer.{0...23}.output.LayerNorm.weight           | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.LayerNorm.bias   | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.key.bias           | UNEXPECTED | 
bert.encoder.layer.{0...23}.intermediate.dense.weight         | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.LayerNorm.weight | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.query.bias         | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.dense.weight     | UNEXPECTED | 
bert.encoder.layer.{0...23}.output.LayerNorm.bias             | UNEXPECTED | 
bert.encoder.layer.{0...23}.output.dense.bias                 | UNEXPECTED | 
crf.end_transitions                                         

📦 Loading weights from ./xlm_roberta_large_checkpointV2/pytorch_model.bin ...


Loading weights: 100%|██████████| 396/396 [00:01<00:00, 260.45tensor/s, loaded=396]


✅ Loaded 396/396 tensors | missing=0 unexpected=0

Running BATCH Test:


/home/slhui.censtatd/.local/lib/python3.12/site-packages/torchcrf/__init__.py:305: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:611.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Original Input : '1M 16 lung sum avenue sheung shui north N.T'
Line 1         : 1M 16 lung sum avenue
Line 2         : sheung shui north N.T
Extracted Tags : {'floor': '1M ', 'building_number': '16 ', 'street_name': 'lung sum avenue ', 'sub_district': 'sheung shui ', 'district': 'north ', 'region': 'N.T'}
Confidences    : {'floor': 0.9977, 'building_number': 0.9999, 'street_name': 0.5516, 'sub_district': 0.7105, 'district': 0.9992, 'region': 0.5422}
Logic Keys Used: ['street_name', 'floor', 'building_number']
Overall Split C: 0.550256   (split logic-aware product)
Status         : SUCCESS
----------------------------------------------------------------------
Original Input : '27LD, Block 5, Hemera, Lohas Park, Lohas Road 1, TKO, HK'
Line 1         : 27LD, Block 5
Line 2         : Hemera, 1, Lohas Park, Lohas Road TKO, HK
Extracted Tags : {'floor': '27LD', 'block': 'Block 5', 'phase': 'Hemera', 'estate_name': 'Lohas Park', 'street_name': 'Lohas Road ', 'building_number': '1', 'sub_distr